# 运行基线 SFT

数据准备和训练配置已在 03.02 中详细解析。现在启动 Wordle SFT 训练。

---

## 训练命令

```bash
NGPU=2 \
MODULE=torchtitan_npu.models.qwen3 \
CONFIG=sft_qwen3_1_7b_wordle \
bash scripts/run_train.sh --checkpoint.last_save_in_hf \
--dataloader.dataset_path ./assets/data/wordle
```

| 参数 | 含义 |
|------|------|
| `NGPU=2` | 使用 2 张 NPU 卡 |
| `MODULE=torchtitan_npu.models.qwen3` | 使用 Qwen3 模型 |
| `CONFIG=sft_qwen3_1_7b_wordle` | 使用 Wordle SFT 配置 |
| `--checkpoint.last_save_in_hf` | 最后一步保存 HuggingFace 格式权重 |

> 关于各配置参数的含义和设计理由（local_batch_size / global_batch_size / gradient accumulation / lr_scheduler / activation_checkpoint / FSDP2），请回顾 **03.02 第 4 节**的详细讲解。

---

## 训练日志解读

训练过程中的关键日志（数值仅用于参考，趋势类似即可）：

```text
step  1: loss 1.25  grad_norm 140.5  tps 392
step  5: loss 0.57  grad_norm   4.0  tps 417
step 10: loss 0.53  grad_norm   3.7  tps 412
step 15: loss 0.50  grad_norm   2.9  tps 405
step 20: loss 0.49  grad_norm   3.0  tps 402
```

| 指标 | 含义 | 健康范围 |
|------|------|--------|
| `loss` | Cross-Entropy Loss，越低越好 | 稳定下降 |
| `grad_norm` | 梯度范数，衡量梯度大小 | 1-10（不爆炸） |
| `tps` | Tokens Per Second，训练吞吐 | 越高越好 |

**训练曲线**：

![SFT Training Curves](./images/training_curves.png)


关键观察：
- Step 1 loss=1.25：基座模型完全不理解 Wordle 格式，loss 较高。
- Step 1→5 loss 1.25→0.57：快速下降（模型迅速学会 `<guess>` 格式）。
- Step 5→20 loss 0.57→0.49：缓慢收敛（继续优化猜测内容）。
- grad_norm 142→3：梯度初始剧烈（模型"震惊"于新数据），迅速稳定。

---

## 训练性能参考

在单卡 Ascend 910B 上的基准数据（Qwen3-1.7B, seq_len=1024）：

| 指标 | 值 |
|------|-----|
| 吞吐量 | ~400 tokens/s |
| MFU | ~1.17% |
| 显存占用 | ~21.85 GB（35%） |
| 每步耗时 | 取决于 batch_size × seq_len |
| 总训练时间 | 20 步约 2-3 分钟 |

> **MFU（Model FLOPs Utilization）**：模型计算利用率。16.5% 看起来不高，这是因为小模型（1.7B）在单卡上受限于内存带宽而非计算能力——第 4 章将通过融合算子提升。

---

## 复制 Tokenizer 文件

训练完成后，将 tokenizer 文件复制到 checkpoint 目录，供推理使用：

```bash
cp assets/hf/Qwen3-1.7B/{config.json,tokenizer.json,tokenizer_config.json,vocab.json,merges.txt} \
  outputs/checkpoint_wordle_sft/step-20/
```

---

## 小结
- 一条命令启动 SFT：`NGPU=1 MODULE=... CONFIG=sft_qwen3_1_7b_wordle bash scripts/run_train.sh`。
- 20 步训练：loss 从 1.25 降至 0.49，grad_norm 从 142 稳定至 ~3。
- 训练完成后保存 HF 格式权重，复制 tokenizer 文件即可推理。

下一节，我们将启动推理服务器，用 vf-eval 评测 SFT 效果。

## 练习

1. （单选题）训练命令中 `--checkpoint.last_save_in_hf` 参数的作用是什么？
    A. 仅保存优化器状态
    B. 最后一步将权重导出为 HuggingFace 格式（含 config.json、model.safetensors）
    C. 每次保存都使用 HuggingFace 格式
    D. 从 HuggingFace 加载权重

2. （单选题）训练日志中 grad_norm 从 step 1 的 142 下降到 step 5 的 4.0，这说明了什么？
    A. 模型训练崩溃
    B. 梯度爆炸
    C. 模型迅速适应新数据分布（学会 `<guess>` 格式后梯度稳定），是 SFT 收敛的正常现象
    D. 学习率设置过低

3. （单选题）MFU 只有 16.5%，主要原因是什么？
    A. NPU 计算单元太少
    B. 小模型（1.7B）在单卡上受限于内存带宽（memory-bound），而非计算能力
    C. 学习率设置过低
    D. batch_size 太小


In [ ]:
!cat ./answer/03.03_answer.txt